# 03. Data Integration: Master Dataset Assembly and Validation

This notebook functions as the Final Integration Gate. It appends the enriched batch data (from Notebook 03) into the master historical repository. This process is the critical point for ensuring data quality, where we perform de-duplication and sanity checks before the dataset is officially committed to the modeling phase.

## Objectives
- Dataset Consolidation: Programmatically merge the newly enriched batch data with the existing master dataset (dataset.csv), ensuring schema alignment across all columns.
- Integrity Enforcement: Identify and purge duplicate records to prevent bias in the predictive model, maintaining a unique source of truth for every flight observation.
- Audit and Quality Assurance: Execute targeted validation scripts on key features (e.g., categorical mappings, boolean holiday triggers) to verify that external enrichment logic was applied correctly and consistently across the updated records.
- Data Governance & Archival: Implement version control for the master dataset by archiving legacy versions before overwriting the production file, maintaining a full audit trail for project reproducibility.

### Environment Setup

In [1]:
import os
import shutil
import pandas as pd
from datetime import datetime

In [2]:
# Path setup
source_dir = './'
dataset_csv = 'dataset.csv'
feature_csv = 'feature_batch.csv'
archive_dir = './dataset_archive'

# Ensure target directories exist dynamically
os.makedirs(archive_dir, exist_ok = True)

### Dataset Consolidation and Archival

In [3]:
# Initialise master storage container
all_flattened_data = []

In [4]:
# Archive current dataset
if os.path.exists(dataset_csv):
    try:
        # Load existing dataset to append them later
        existing_df = pd.read_csv(dataset_csv)
        all_flattened_data = existing_df.to_dict(orient='records')
        print(f"Loaded {len(all_flattened_data)} existing records from {dataset_csv}")
    except Exception as e:
        print(f"Warning: Could not read {dataset_csv} ({e}). Starting fresh.")
    
    # Archiving dataset and adding today's date
    today_str = datetime.now().strftime('%Y-%m-%d')
    archive_file_name = f"dataset_{today_str}.csv"
    archive_path = os.path.join(archive_dir, archive_file_name)

    # Move the current file into the archive folder
    shutil.move(dataset_csv, archive_path)
    print(f"Sucessfully archived old file to: {archive_path}")

else:
    if not os.path.exists(dataset_csv):
        print(f"Error: Could not locate {dataset_csv}")

Loaded 1524 existing records from dataset.csv
Sucessfully archived old file to: ./dataset_archive\dataset_2026-05-29.csv


In [5]:
# Load feature batch
batch_df = pd.read_csv(feature_csv)

print(f"Loaded {len(batch_df)} records from {feature_csv}.")

Loaded 308 records from feature_batch.csv.


In [6]:
# Append feature batch to the dataset
combined_df = pd.concat([existing_df, batch_df], axis=0, ignore_index = True)

### Integrity Enforcement

**Drivers of Data Redundancy:**
- Flight Frequency: Airlines often have multiple flights per day for the same route and date. Because our data doesn't include specific flight numbers, these different flights look like "noisy" duplicates.
- Extraction Overlap: Re-running our data collection script can sometimes pull the same flight info multiple times, creating redundant entries.

**Deduplication Strategy and Rationale:**
- Preventing Bias: If we kept all duplicates, the model would over-emphasize specific flights, causing it to "overfit" (memorize data) rather than learn general pricing trends.
- Ensuring Accuracy: This standardizes our data, so each row in our training set represents one unique market offering. This helps the model map pricing patterns more cleanly.
- Data Integrity: This creates a clean "source of truth," ensuring the model learns from actual market fluctuations rather than technical noise from our extraction process.
- Better Performance: By reducing the dataset size, we save memory and significantly speed up the time it takes to train and test our model.

In [7]:
# Remove duplicate records if cells are executed twice. 
dedup_keys = ['date', 'route', 'departure_date', 'airline', 'price']

# Check if these keys exist in the combined dataset
available_keys = [key for key in dedup_keys if key in combined_df.columns]

initial_count = len(combined_df)
if available_keys:
    combined_df = combined_df.drop_duplicates(subset=available_keys, keep='first')
else:
    combined_df = combined_df.drop_duplicates()

dropped_rows = initial_count - len(combined_df)
if dropped_rows > 0: 
    print(f"Dropped {dropped_rows} duplicate rows")

Dropped 112 duplicate rows


### Data Export

In [8]:
# Save new dataset
combined_df.to_csv(dataset_csv, index=False)
print(f"Saved new dataset at {dataset_csv}")

Saved new dataset at dataset.csv


### Audit and Quality Assurance

In [9]:
# View unique values to see if there are any anomolies from earlier steps.

unique_column = [
    'route',
    'is_weekend',
    'departure_airport',
    'out_inbound',
    'other_airport',
    'data_source',
    'booking_window',
    'airline',
    'airline_code',
    'is_lcc',
    'is_holiday_sin',
    'is_holiday_other',
    'is_sch_holiday'
]

for col in unique_column:
    if col in combined_df.columns:
        unique_vals = combined_df[col].dropna().unique()

        try:
            unique_vals = sorted(unique_vals)
        except TypeError:
            pass

        print(f"Column: {col} | Total distinct elements: {len(unique_vals)}")
        print(f"   {list(unique_vals)}")

    else: 
        print(f"Column: {col} was not found.")

Column: route | Total distinct elements: 8
   ['BKK-SIN', 'LHR-SIN', 'MEL-SIN', 'NRT-SIN', 'SIN-BKK', 'SIN-LHR', 'SIN-MEL', 'SIN-NRT']
Column: is_weekend | Total distinct elements: 2
   [np.False_, np.True_]
Column: departure_airport | Total distinct elements: 5
   ['BKK', 'LHR', 'MEL', 'NRT', 'SIN']
Column: out_inbound | Total distinct elements: 2
   [np.int64(1), np.int64(2)]
Column: other_airport | Total distinct elements: 4
   ['BKK', 'LHR', 'MEL', 'NRT']
Column: data_source | Total distinct elements: 2
   ['api', 'manual']
Column: booking_window | Total distinct elements: 6
   ['1-14 days', '15-28 days', '29-42 days', '43-56 days', '57-70 days', '71-84 days']
Column: airline | Total distinct elements: 11
   ['ANA', 'British Airways', 'JAL', 'Jetstar', 'NIL', 'Qantas', 'Scoot', 'Singapore Airlines', 'THAI', 'Turkish Airlines', 'ZIPAIR Tokyo']
Column: airline_code | Total distinct elements: 11
   ['BA', 'JL', 'JQ', 'NH', 'NIL', 'QF', 'SQ', 'TG', 'TK', 'TR', 'ZG']
Column: is_lcc | To